In [1]:
# imports

# to get my token from .env
from dotenv import load_dotenv

# get token
import os

# handle requests 
import requests 

In [2]:
# get token from .env

# load .env
load_dotenv()
# get token
TOKEN = os.getenv("GITHUB_TOKEN")

# ensure TOKEN exists
print(TOKEN is not None)

True


In [3]:
# declare headers with TOKEN

headers = {
    "Authorization": f"Bearer {TOKEN}"
}

In [4]:
try:
    # get users after id 0
    since = 0
    # number of pages to ingest
    pages = 2
    
    # store user data
    all_users_data = []

    # store repos data
    all_repos_data = []

    # store repo commits data
    all_commits_data = []

    for i in range(pages):
        
        # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        # SET TO PER_PAGE=30 FOR TESTING PURPOSES
        # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        
        # per_page is set to 100 to get a 100 users per call
        url = f"https://api.github.com/users?since={since}&per_page=30"
        r = requests.get(url, headers=headers)

        # check if there is an error
        r.raise_for_status()

        # store json data
        users = r.json()

        # break if users is empty list > stop pagination loop
        if not users:
            break

        # get the usernames to get more data
        logins = [user["login"] for user in users]

        for login in logins:
            # url for specific user data using username
            url = f"https://api.github.com/users/{login}"
            user_r = requests.get(url, headers=headers)

            user_r.raise_for_status()

            # returns a dict per user
            user_data = user_r.json()

            # append user dict to all data
            all_users_data.append(user_data)

            # repo section
            # get 10 repos per user
            url = f"https://api.github.com/users/{login}/repos?per_page=10"
            repo_r = requests.get(url, headers=headers)

            repo_r.raise_for_status()

            # returns a list
            repo_data = repo_r.json()

            # extend repos data list to include repo data
            all_repos_data.extend(repo_data)
            
            # for each repo get the commits for that repo
            for repo in repo_data:

                # get repo name for url
                repo_name = repo["name"]

                url = f"https://api.github.com/repos/{login}/{repo_name}/commits?per_page=20"
                commit_r = requests.get(url, headers=headers)
                
                if commit_r.status_code == 409:
                    print(commit_r.status_code)
                    # commit history not accessible
                    print(commit_r)
                    continue
                # check for other errors 
                else:
                    commit_r.raise_for_status()

                # returns a list
                commit_data = commit_r.json()

                all_commits_data.extend(commit_data)
        
        # GitHub REST api with users endpoint uses 'since' to get the users where the id is after 'since'
        # therefore, get the last id of the current call to get the next page
        since = users[-1]["id"]

# handle HTTP status errors first, then catch other request failures
except requests.exceptions.HTTPError as e:
    print("HTTP error:", e)
    
except requests.exceptions.RequestException as e:
    print("Other request error:", e)

409
<Response [409]>
409
<Response [409]>
409
<Response [409]>
409
<Response [409]>


In [5]:
# different endpoints
#/users/{username}/repos
#/repos/{owner}/{repo}/commits
#/repos/{owner}/{repo}/issues

In [6]:
print(all_repos_data[0]["name"])

30daysoflaptops.github.io
